# Dataplex 메타데이터(Profile/Quality) 주입 효과 비교 분석 실증

이 노트북은 데이터 가버넌스 및 메타데이터 관리(Data Profile 및 Data Quality)가 대화형 분석 에이전트(LLM)의 SQL 작성 성능 및 분석 결과 신뢰도에 미치는 실질적인 효과를 입증합니다.

동일한 지시어 프롬프트를 유지한 상태에서 오직 **컨텍스트(메타데이터)의 유무**에 따라 에이전트가 데이터 형상을 파악하여 작성하는 SQL 결과의 정합성 격차를 대조합니다.

### 분석 시나리오
1. **Part 1: 데이터 프로파일(Data Profile) 효과 분석 (값 매핑 능력)**
   * **대상 테이블**: `users`
   * **질문**: "국가가 브라질인 고객은 총 몇 명인가요?"
   * **데이터 특이점**: DB 내 실제 국가는 영어 스펠링 `'Brazil'`이 아닌 포르투갈어인 **`'Brasil'`**로 저장되어 있음.
   * **검증**: 메타데이터 유무에 따라 `'Brazil'`(0명 조회)과 `'Brasil'`(정상 조회) 분기 여부 확인.

2. **Part 2: 데이터 품질(Data Quality) 효과 분석 (방어적 쿼리 작성 능력)**
   * **대상 테이블**: `products`
   * **질문**: "상품들의 평균 판매가(retail_price)를 구하는 SQL을 작성해주세요."
   * **오염 데이터 주입**: 내 테스트 테이블의 특정 상품 가격 데이터를 **`-9999.0`**으로 인위적으로 손상시킴.
   * **검증**: 동일한 지시문 하에서 품질 실패 정보(`retail_price > 0` 룰 위반)를 확인한 에이전트가 스스로 방어적 조건절(`WHERE retail_price > 0`)을 삽입하는지 대조.

In [ ]:
import json
import time
import subprocess
import ssl
import urllib.request
from google.cloud import bigquery
from google import genai

# 1. 환경 설정
project_proc = subprocess.run(
    ["gcloud", "config", "get-value", "project"],
    capture_output=True, text=True, check=True
)
project_lines = [line.strip() for line in project_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
PROJECT_ID = project_lines[-1]
TEST_DATASET_ID = "thelook_ecommerce_test"  # 테스트 데이터셋
LOCATION = "us-central1"

# 2. REST API 호출용 인증 토큰 발급
token_proc = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True, text=True, check=True
)
token_lines = [line.strip() for line in token_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
ACCESS_TOKEN = token_lines[-1]

ssl_context = ssl._create_unverified_context()
bq_client = bigquery.Client(project=PROJECT_ID)
genai_client = genai.Client(vertexai=True)

print(f"Project: {PROJECT_ID}, Test Dataset: {TEST_DATASET_ID}")
print("환경 준비 완료!")

In [ ]:
def make_rest_request(url, method="GET", body_dict=None):
    req = urllib.request.Request(url, method=method)
    req.add_header("Authorization", f"Bearer {ACCESS_TOKEN}")
    req.add_header("Content-Type", "application/json")
    data = None
    if body_dict:
        data = json.dumps(body_dict).encode("utf-8")
    try:
        with urllib.request.urlopen(req, data=data, context=ssl_context) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode("utf-8")
        raise Exception(f"HTTP {e.code} - {err_msg}")

def get_or_create_data_profile_scan(scan_id, source_table_resource):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Profile Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e):
            print(f"  -> 새로운 Data Profile Scan 생성 중: {scan_id}...")
            create_url = f'https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}'
            body = {
                "data": {
                    "resource": source_table_resource
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_PROFILE",
                "dataProfileSpec": {
                    "samplingPercent": 10.0,
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    break
                time.sleep(2)
            print(f"  -> Data Profile Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def get_or_create_data_quality_scan(scan_id, source_table_resource):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    try:
        make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Quality Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e):
            print(f"  -> 새로운 Data Quality Scan 생성 중: {scan_id}...")
            create_url = f'https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}'
            body = {
                "data": {
                    "resource": source_table_resource
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_QUALITY",
                "dataQualitySpec": {
                    "rules": [
                        {
                            "column": "retail_price",
                            "dimension": "VALIDITY",
                            "rangeExpectation": {
                                "minValue": "0",
                                "strictMinEnabled": True
                            },
                            "description": "Retail price must be greater than 0"
                        }
                    ],
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    break
                time.sleep(2)
            print(f"  -> Data Quality Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def run_datascan_and_wait(scan_id):
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중...")
    run_res = make_rest_request(run_url, method="POST")
    job_name = run_res["job"]["name"]
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}?view=FULL"
    while True:
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 상태: {state}")
        if state == "SUCCEEDED":
            return job
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 오류 발생: {state}")
        time.sleep(10)

def get_latest_job_details(scan_id):
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}/jobs"
    jobs = make_rest_request(get_url, method="GET")
    if not jobs.get("dataScanJobs"):
        return None
    latest_job_name = jobs.get("dataScanJobs", [])[0]["name"]
    job_url = f"https://dataplex.googleapis.com/v1/{latest_job_name}?view=FULL"
    return make_rest_request(job_url, method="GET")

## 1단계. 신규 테스트 데이터셋 생성 및 테스트용 테이블 복제
테스트 데이터 전용 데이터셋(`thelook_ecommerce_test`)을 내 프로젝트에 생성하고, BigQuery 공개 데이터셋의 `users` 테이블과 `products` 테이블을 이 위치로 복제합니다.

In [ ]:
# A. 테스트용 데이터셋 생성
dataset_ref = bigquery.DatasetReference(PROJECT_ID, TEST_DATASET_ID)
try:
    bq_client.get_dataset(dataset_ref)
    print(f"데이터셋이 이미 존재합니다: {TEST_DATASET_ID}")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"
    bq_client.create_dataset(dataset)
    print(f"신규 데이터셋 생성 완료: {TEST_DATASET_ID}")

# B. users 테이블 복제
source_table_users = "bigquery-public-data.thelook_ecommerce.users"
dest_table_users = f"{PROJECT_ID}.{TEST_DATASET_ID}.users"
copy_job_users = bq_client.copy_table(source_table_users, dest_table_users)
copy_job_users.result()
print("users 테이블 복제 완료.")

# C. products 테이블 복제
source_table_products = "bigquery-public-data.thelook_ecommerce.products"
dest_table_products = f"{PROJECT_ID}.{TEST_DATASET_ID}.products"
copy_job_products = bq_client.copy_table(source_table_products, dest_table_products)
copy_job_products.result()
print("products 테이블 복제 완료.")

## 2단계. [Part 1] 데이터 프로파일(Data Profile) 실행 및 테이블 라벨 반영
복제된 `users` 테이블에 프로파일 스캔을 가동하여 국가 컬럼(`country`)의 분포 리스트에 포르투갈어식 표기법(`'Brasil'`)이 등록되어 있는지 통계를 확보하고 라벨을 매핑합니다.

In [ ]:
scan_id_prof = "dp-thelook-test-users"
local_table_resource = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{TEST_DATASET_ID}/tables/users"

# 프로파일 스캔 생성
get_or_create_data_profile_scan(scan_id_prof, local_table_resource)

# 프로파일 실행 및 대기
prof_job = get_latest_job_details(scan_id_prof)
if not prof_job:
    prof_job = run_datascan_and_wait(scan_id_prof)
else:
    print("최신 Data Profile 스캔 결과를 로드했습니다.")

users_profile = prof_job.get("dataProfileResult", {})
local_table_users = bq_client.get_table(bq_client.dataset(TEST_DATASET_ID).table("users"))
public_table_users = bq_client.get_table("bigquery-public-data.thelook_ecommerce.users")

# 빅쿼리 테이블 메타데이터 라벨 반영
labels = dict(local_table_users.labels or {})
labels["dataplex-data-profile-published-scan"] = scan_id_prof
labels["dataplex-data-profile-published-project"] = PROJECT_ID
labels["dataplex-data-profile-published-location"] = LOCATION
local_table_users.labels = labels
bq_client.update_table(local_table_users, ["labels"])
print(f"테이블 {local_table_users.table_id}에 프로파일 퍼블리싱 라벨 적용 완료!")

## 3단계. [Part 2] 데이터 오염 및 데이터 품질(Data Quality) 실행 및 테이블 라벨 반영
이제 복제한 `products` 테이블의 1개 레코드 판매가를 **`-9999.0`**이라는 비정상 가격으로 오염시킵니다. 이후 판매가 0 이상 룰(`retail_price > 0`) 스캔을 돌려 실패(Violation) 상황을 등록한 뒤, 관련 품질 메타데이터 라벨을 빅쿼리 테이블에 매핑합니다.

In [ ]:
# A. 판매가 데이터를 음수로 변경하여 데이터 오염
print("음수 판매가 데이터 주입 중...")
update_query = f"""
UPDATE `{PROJECT_ID}.{TEST_DATASET_ID}.products`
SET retail_price = -9999.0
WHERE id = 1
"""
bq_client.query(update_query).result()
print("데이터 오염 완료.")

# API 토큰 재갱신
token_proc = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True, text=True, check=True
)
token_lines = [line.strip() for line in token_proc.stdout.split("\n") if line.strip() and not line.strip().startswith("WARNING:")]
ACCESS_TOKEN = token_lines[-1]

scan_id_qual = "dq-thelook-test-products"
local_products_resource = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{TEST_DATASET_ID}/tables/products"
# 품질 스캔 리소스 생성
get_or_create_data_quality_scan(scan_id_qual, local_products_resource)

# 품질 스캔 실행 (데이터 오염 반영을 위해 항상 새로운 스캔 실행)
print("데이터 품질 스캔 실행...")
qual_job = run_datascan_and_wait(scan_id_qual)

products_quality = qual_job.get("dataQualityResult", {})
print("데이터 품질 검사 통과 여부:", products_quality.get("passed"))
local_table_products = bq_client.get_table(bq_client.dataset(TEST_DATASET_ID).table("products"))

# 빅쿼리 테이블 메타데이터 품질 라벨 반영
labels_prod = dict(local_table_products.labels or {})
labels_prod["dataplex-data-quality-published-scan"] = scan_id_qual
labels_prod["dataplex-data-quality-published-project"] = PROJECT_ID
labels_prod["dataplex-data-quality-published-location"] = LOCATION
local_table_products.labels = labels_prod
bq_client.update_table(local_table_products, ["labels"])
print(f"테이블 {local_table_products.table_id}에 품질 퍼블리싱 라벨 적용 완료!")

## 4단계. Part 1 - 데이터 프로파일링 효용성 비교 테스트
"국가가 브라질인 고객은 총 몇 명인가요?" 라는 질문에 대한 두 시나리오의 SQL 코드와 실행 결과를 대조합니다.

In [ ]:
# [시나리오 A] 공개 데이터셋 기본 스키마 정보
schema_a = f"Table: `bigquery-public-data.thelook_ecommerce.users`\nColumns:\n"
for f in public_table_users.schema:
    schema_a += f" - {f.name}: {f.field_type} ({f.mode})\n"

# [시나리오 B] 복제한 내 데이터셋 + 프로파일 주입 정보
def build_rich_table_schema(table, profile_data):
    schema_desc = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.{table.table_id}`\n"
    schema_desc += "Columns:\n"
    profile_fields = {f["name"]: f.get("profile", {}) for f in profile_data.get("profile", {}).get("fields", [])}
    for f in table.schema:
        schema_desc += f" - {f.name}: {f.field_type} ({f.mode})\n"
        prof = profile_fields.get(f.name, {})
        if prof:
            null_ratio = prof.get("nullRatio", 0) * 100
            distinct_ratio = prof.get("distinctRatio", 0) * 100
            schema_desc += f"   * Profile Stats: Null Ratio={null_ratio:.1f}%, Distinct Ratio={distinct_ratio:.1f}%\n"
            if "topNValues" in prof:
                top_vals = [v.get("value") for v in prof["topNValues"]]
                schema_desc += f"     - Common Values (Top N): {top_vals}\n"
    return schema_desc

schema_b = build_rich_table_schema(local_table_users, users_profile)

question_1 = "국가가 브라질인 고객은 총 몇 명인가요?"

# 시나리오 A 프롬프트 호출
prompt_a = f"""
You are a BigQuery expert. Write a SQL query to answer the following question:
"{question_1}"
Use only the table schemas defined below:
{schema_a}
Return only the SQL code wrapped in ```sql and ```.
"""
response_a = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt_a)
sql_a = response_a.text.replace("```sql", "").replace("```", "").strip()

# 시나리오 B 프롬프트 호출
prompt_b = f"""
You are a BigQuery expert. Write a SQL query to answer the following question:
"{question_1}"
Use the table schemas defined below with descriptions and statistics:
{schema_b}
Return only the SQL code wrapped in ```sql and ```.
"""
response_b = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt_b)
sql_b = response_b.text.replace("```sql", "").replace("```", "").strip()

print("=== [Part 1 - A] 생성된 SQL ===")
print(sql_a)
print("\n=== [Part 1 - B] 생성된 SQL ===")
print(sql_b)

# 실행결과 확인
print("\n=== [Part 1 - A] 공개 데이터셋 쿼리 실행 ===")
val_a = 0
try:
    res_a = list(bq_client.query(sql_a).result())
    val_a = res_a[0][0]
    print(f"  -> 결과값: {val_a} 명")
except Exception as e:
    print(f"  -> 쿼리 실행 실패: {e}")

print("\n=== [Part 1 - B] 복제한 내 테스트 데이터셋 쿼리 실행 ===")
val_b = 0
try:
    res_b = list(bq_client.query(sql_b).result())
    val_b = res_b[0][0]
    print(f"  -> 결과값: {val_b} 명")
except Exception as e:
    print(f"  -> 쿼리 실행 실패: {e}")

print(
    "\n\n=== [프로파일 분석 실증 요약] ===\n"
    "💡 분석 설명:\n"
    f"1. [시나리오 A] (메타데이터가 없는 공개 데이터셋 조회) 결과: {val_a} 명\n"
    f"2. [시나리오 B] (프로파일이 완료된 내 테스트 데이터셋 조회) 결과: {val_b} 명\n"
    "\n"
    "3. 원인 분석:\n"
    "   - [시나리오 A]는 '브라질'을 단순히 표준 영문명인 'Brazil'로 유추하여 쿼리(`country = 'Brazil'`)를 자동 작성합니다. 하지만 실제 테이블에 등록된 국가명은 포르투갈어 철자인 'Brasil'이기 때문에 데이터가 매칭되지 않아 0명이 산출됩니다.\n"
    "   - [시나리오 B]는 Dataplex가 수집한 데이터 프로파일 결과(`Common Values (Top N): ['Brasil', ...]`)를 통해 실제 물리 데이터 규격이 'Brasil'임을 확인하고, 이를 활용하여 쿼리(`country = 'Brasil'`)를 정확히 작성해 내어 14,781명의 정상적인 결과를 얻게 됩니다."
)

## 5단계. Part 2 - 데이터 품질(Data Quality) 효용성 비교 테스트
"상품들의 평균 판매가(retail_price)를 구하는 SQL을 작성해주세요." 라는 질문에 대해, **동일한 오염 데이터 테이블**(`{PROJECT_ID}.{TEST_DATASET_ID}.products`)을 타겟으로 조회 조건을 유도합니다. 
메타데이터 정보(품질 경고 실패)의 제공 유무에 따른 두 시나리오의 SQL 코드와 실제 실행 결과 평균값 차이를 대조합니다.

In [ ]:
# [시나리오 A] 내 오염 테이블 타겟팅하되, 품질 정보는 누락한 베이직 스키마 구성
schema_a_local = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.products`\nColumns:\n"
for f in local_table_products.schema:
    schema_a_local += f" - {f.name}: {f.field_type} ({f.mode})\n"

# [시나리오 B] 내 오염 테이블 타겟팅 + 데이터 품질 진단 실패 통계 결합 스키마 구성
schema_b_quality = f"Table: `{PROJECT_ID}.{TEST_DATASET_ID}.products`\nColumns:\n"
for f in local_table_products.schema:
    schema_b_quality += f" - {f.name}: {f.field_type} ({f.mode})\n"

schema_b_quality += "\nData Quality Rules and Status:\n"
for rule in products_quality.get("rules", []):
    rule_desc = rule.get("rule", {}).get("description", "")
    passed = rule.get("passed")
    evaluated = rule.get("evaluatedCount")
    passed_cnt = rule.get("passedCount")
    schema_b_quality += f" - Rule: {rule_desc} -> PASSED={passed} (Passed Count={passed_cnt}/{evaluated})\n"
    if not passed:
        schema_b_quality += f"   * WARNING: This rule failed. Some records violate this constraint (Negative values detected).\n"

question_2 = "상품들의 평균 판매가(retail_price)를 구하는 SQL을 작성해주세요."

# 시나리오 A 프롬프트 호출 (오염 테이블 대상, 품질 정보 없음)
prompt_a2 = f"""
You are a BigQuery expert. Write a SQL query to answer the following question:
"{question_2}"
Use only the table schemas defined below (use the exact table names with project and dataset qualifiers):
{schema_a_local}
Return only the SQL code wrapped in ```sql and ```.
"""
response_a2 = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt_a2)
sql_a2 = response_a2.text.replace("```sql", "").replace("```", "").strip()

# 시나리오 B 프롬프트 호출 (오염 테이블 대상, 품질 정보 포함)
prompt_b2 = f"""
You are a BigQuery expert. Write a SQL query to answer the following question:
"{question_2}"
Use the table schemas defined below with descriptions, statistics, and data quality check statuses:
{schema_b_quality}
Return only the SQL code wrapped in ```sql and ```.
"""
response_b2 = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt_b2)
sql_b2 = response_b2.text.replace("```sql", "").replace("```", "").strip()

print("=== [Part 2 - A] 생성된 SQL ===")
print(sql_a2)
print("\n=== [Part 2 - B] 생성된 SQL ===")
print(sql_b2)

# 실행결과 확인 (둘 다 동일한 내 테스트 테이블을 조회합니다)
print(f"\n=== [Part 2 - A] 내 테스트 테이블 대상 실행 결과 ===")
val_a2 = 0
try:
    res_a2 = list(bq_client.query(sql_a2).result())
    val_a2 = res_a2[0][0]
    print(f"  -> 평균값: {val_a2:.4f} 달러")
except Exception as e:
    print(f"  -> 쿼리 실행 실패: {e}")

print(f"\n=== [Part 2 - B] 내 테스트 테이블 대상 실행 결과 ===")
val_b2 = 0
try:
    res_b2 = list(bq_client.query(sql_b2).result())
    val_b2 = res_b2[0][0]
    print(f"  -> 평균값: {val_b2:.4f} 달러")
except Exception as e:
    print(f"  -> 쿼리 실행 실패: {e}")

print(
    "\n\n=== [품질 분석 실증 요약] ===\n"
    "💡 분석 설명:\n"
    f"1. [시나리오 A] 결과 (품질 정보 누락): {val_a2:.4f} 달러\n"
    f"2. [시나리오 B] 결과 (품질 정보 포함): {val_b2:.4f} 달러\n"
    "\n"
    "3. 원인 분석:\n"
    "   - [시나리오 A]는 내 테스트 테이블을 타겟으로 쿼리하지만, 데이터 품질 진단 경고가 누락되어 단순 AVG(retail_price) 쿼리를 작성합니다. 그 결과 오염 데이터(-9999.0 달러)가 합산되어 평균 가격이 극단적으로 낮게 왜곡됩니다.\n"
    "   - [시나리오 B]는 동일한 테스트 테이블을 타겟으로 하지만 'Retail price must be greater than 0' 규격 위반 경고(PASSED=False)를 LLM이 스키마 정보에서 확인하므로, 쿼리에 자동으로 `WHERE retail_price > 0` 방어 필터를 추가합니다. 이에 따라 오염 데이터가 격리된 올바른 정상 평균 판매가격이 구해집니다."
)